# ROS Bootcamp — Day 2 (ROS2 Basics + Vision)
This notebook has two parts:
1. **ROS2 basics**: publisher/subscriber, topics, messages, services (client/server)
2. **Vision over ROS2**: publish camera images on a topic, subscribe and run **YOLO + InsightFace**, publish results

You will run the ROS2 nodes from a terminal (not inside the notebook).

Assumption: You are using **ROS2 Humble** on Ubuntu 22.04 (or similar).

## 0) ROS2 setup checklist
### 0.1 Source ROS2
```bash
source /opt/ros/humble/setup.bash
```

### 0.2 Create a workspace
```bash
mkdir -p ~/ros_bootcamp_ws/src
cd ~/ros_bootcamp_ws
```

### 0.3 Install ROS2 Python deps
```bash
sudo apt update
sudo apt install -y python3-colcon-common-extensions \
  ros-humble-rclpy ros-humble-std-msgs ros-humble-sensor-msgs \
  ros-humble-cv-bridge ros-humble-image-transport \
  ros-humble-turtlesim
```

### 0.4 Python ML deps (same as Day 1)
Inside your Python env:
```bash
pip install opencv-python numpy insightface onnxruntime ultralytics
```

> If you use a venv/conda: make sure your ROS2 python can see it, or run nodes with that python.

## 1) ROS2 basics — mental model
- **Node**: a process that does one job.
- **Topic**: a named channel that carries a stream of messages.
- **Publisher**: sends messages to a topic.
- **Subscriber**: receives messages from a topic.
- **Message type**: the schema of data on the wire (e.g., `std_msgs/String`, `sensor_msgs/Image`).
- **Service (client/server)**: request-response (like a function call over the network).

We’ll implement one minimal example for each:
1. String talker/listener (pub/sub)
2. AddTwoInts service (client/server)
3. Camera image publisher (Image topic)
4. Vision subscriber node (YOLO + InsightFace) that publishes results
5. Quick sim: `turtlesim` (subscribe to pose / publish to cmd_vel)

## 2) Codebase layout (we provide it)
We provide a ready-to-build workspace zip with two packages:

- `bootcamp_basics`:
  - `talker.py` publishes `std_msgs/String` on `/chatter`
  - `listener.py` subscribes `/chatter`
  - `add_two_ints_srv.py` service server on `/add_two_ints`
  - `add_two_ints_client.py` service client
  - `turtlesim_square.py` publishes velocity commands

- `bootcamp_vision`:
  - `camera_pub.py` publishes `sensor_msgs/Image` on `/camera/image_raw`
  - `vision_node.py` subscribes to `/camera/image_raw`, runs YOLO + InsightFace, publishes:
      - annotated image on `/vision/annotated`
      - JSON string detections on `/vision/detections`

You will build with `colcon` and run with `ros2 run`.

Next, see step-by-step commands below.

## 3) Step-by-step: build & run
### 3.1 Unzip the workspace
Put the folder in your home directory:
```bash
unzip ros_bootcamp_ws.zip -d ~/
```

You will get: `~/ros_bootcamp_ws/src/...`

### 3.2 Build
```bash
cd ~/ros_bootcamp_ws
colcon build
source install/setup.bash
```

### 3.3 Run ROS2 basics examples
Terminal A:
```bash
source /opt/ros/humble/setup.bash
cd ~/ros_bootcamp_ws && source install/setup.bash
ros2 run bootcamp_basics talker
```

Terminal B:
```bash
source /opt/ros/humble/setup.bash
cd ~/ros_bootcamp_ws && source install/setup.bash
ros2 run bootcamp_basics listener
```

### 3.4 Service example
Terminal A:
```bash
ros2 run bootcamp_basics add_two_ints_srv
```
Terminal B:
```bash
ros2 run bootcamp_basics add_two_ints_client 7 35
```

### 3.5 Vision over ROS2
Terminal A (camera publisher):
```bash
ros2 run bootcamp_vision camera_pub
```
Terminal B (vision node):
```bash
ros2 run bootcamp_vision vision_node
```

Optional view tools:
```bash
ros2 topic list
ros2 topic echo /vision/detections
rqt_image_view
```

### 3.6 Turtlesim mini-sim
Terminal A:
```bash
ros2 run turtlesim turtlesim_node
```
Terminal B:
```bash
ros2 run bootcamp_basics turtlesim_square
```

## 4) How the vision pipeline works (ROS version)
1. `camera_pub.py` reads webcam frames with OpenCV.
2. It converts frames to `sensor_msgs/Image` using `cv_bridge`.
3. It publishes on `/camera/image_raw`.
4. `vision_node.py` subscribes to `/camera/image_raw`.
5. It runs:
   - YOLO: bounding boxes for objects
   - InsightFace: faces + embeddings + recognition (optional DB)
6. It publishes:
   - `/vision/annotated` (Image) for visualization
   - `/vision/detections` (String JSON) for simple downstream consumption

This is the same idea as Day 1, but **decoupled** using ROS topics.

## 5) Troubleshooting
- If `camera_pub` can't open camera: change `CAM_ID` in the node.
- If `cv_bridge` import fails: confirm `ros-humble-cv-bridge` installed.
- If InsightFace model download fails: check internet access and rerun.
- If nodes don't see each other:
  - confirm you sourced both `/opt/ros/...` and the workspace `install/setup.bash`
  - ensure `ROS_DOMAIN_ID` matches across terminals (default is fine).
- If performance is slow: lower camera resolution or use `yolov8n.pt` and set `conf` higher.
